Mengimpor library yang dibutuhkan

In [ ]:
!pip install PySastrawi

import pandas as pd
import re
from google.colab import drive
drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.6/210.6 kB 153.0 kB/s eta 0:00:00
Mounted at /content/drive


Gathering Data

In [ ]:
!ls /content/drive/MyDrive/dataset_berita_id.csv

/content/drive/MyDrive/dataset_berita_id.csv


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/dataset_berita_id.csv')
print(f"Data awal: {len(df)} baris")

Data awal: 80472 baris


In [ ]:
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80472 entries, 0 to 80471
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Judul    80472 non-null  object
 1   Waktu    80472 non-null  object
 2   Link     80472 non-null  object
 3   Content  80463 non-null  object
 4   tag1     78023 non-null  object
 5   tag2     77612 non-null  object
 6   tag3     71602 non-null  object
 7   tag4     56856 non-null  object
 8   tag5     35633 non-null  object
 9   source   80472 non-null  object
dtypes: object(10)
memory usage: 6.1+ MB


In [ ]:
df.describe()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
count,80472,80472,80472,80463,78023,77612,71602,56856,35633,80472
unique,67168,26970,67381,67367,18490,24482,26226,24944,18463,3
top,5 Berita Terpopuler Internasional Hari Ini,26 November 2024,https://news.detik.com/berita/d-8058078/menter...,"Baca berita dengan sedikit iklan, klik di sin...",Jakarta,jabodetabek,jabodetabek,jabodetabek,jabodetabek,kompas
freq,7,149,4,5,865,488,915,951,557,35449


In [ ]:
df.isnull().sum()

,0
Judul,0
Waktu,0
Link,0
Content,9
tag1,2449
tag2,2860
tag3,8870
tag4,23616
tag5,44839
source,0


In [ ]:
link_contoh = df[df.duplicated(subset=["Link"], keep=False)]["Link"].iloc[0]
print("Link:", link_contoh)
print()
print(df[df["Link"] == link_contoh][["Judul", "Waktu", "source"]])

Link: https://news.detik.com/berita/d-7486691/gempa-m-5-3-guncang-pulau-doi-maluku-utara

                                            Judul  \
1      Gempa M 5,3 Guncang Pulau Doi Maluku Utara   
6163   Gempa M 5,3 Guncang Pulau Doi Maluku Utara   
73329  Gempa M 5,3 Guncang Pulau Doi Maluku Utara   

                              Waktu source  
1      Senin, 12 Agu 2024 21:58 WIB  detik  
6163   Senin, 12 Agu 2024 21:58 WIB  detik  
73329  Senin, 12 Agu 2024 21:58 WIB  detik  


In [ ]:
duplikat_count = df[df.duplicated(subset=["Link"], keep=False)].groupby("Link").size()

print("Artikel muncul 2x:", (duplikat_count == 2).sum())
print("Artikel muncul 3x:", (duplikat_count == 3).sum())
print("Artikel muncul >3x:", (duplikat_count > 3).sum())
print("Total link unik yang duplikat:", len(duplikat_count))

Artikel muncul 2x: 308
Artikel muncul 3x: 6366
Artikel muncul >3x: 17
Total link unik yang duplikat: 6691


In [ ]:
df = df.drop_duplicates(subset=["Link"])
print("Data bersih:", len(df), "baris")

Data bersih: 67381 baris


In [ ]:
tag_cols = ['tag1', 'tag2', 'tag3', 'tag4', 'tag5']
df['all_tags'] = df[tag_cols].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)

In [ ]:
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source,all_tags
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas,"Said Iqbal, industri rokok, PT Gudang Garam, P..."
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik,"pulau doi, gempa"
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik,"pemalsuan emas, emas palsu, polres bengkalis, ..."
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik,"minyakita, kudus"
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo,"jepang, perdana-menteri, sanae-takaichi, perem..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67381 entries, 0 to 80471
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Judul     67381 non-null  object
 1   Waktu     67381 non-null  object
 2   Link      67381 non-null  object
 3   Content   67378 non-null  object
 4   tag1      64939 non-null  object
 5   tag2      64725 non-null  object
 6   tag3      60423 non-null  object
 7   tag4      49805 non-null  object
 8   tag5      32264 non-null  object
 9   source    67381 non-null  object
 10  all_tags  67381 non-null  object
dtypes: object(11)
memory usage: 6.2+ MB


In [ ]:
def bersihkan_teks(teks):
    if pd.isna(teks):
        return ""
    teks = str(teks).lower() # Mengubah teks menjadi huruf kecil
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE) # Menghapus link URL
    teks = re.sub(r'[^\w\s]', ' ', teks) # Menghapus tanda baca
    teks = re.sub(r'\s+', ' ', teks).strip() # Menghapus spasi berlebih
    return teks

# Mengaplikasikan fungsi pembersihan ke kolom yang akan dianalisis
df['clean_tags'] = df['all_tags'].apply(bersihkan_teks)
df['clean_judul'] = df['Judul'].apply(bersihkan_teks)
df['clean_konten'] = df['Content'].apply(bersihkan_teks)

In [ ]:
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source,all_tags,clean_tags,clean_judul,clean_konten
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas,"Said Iqbal, industri rokok, PT Gudang Garam, P...",said iqbal industri rokok pt gudang garam phk ...,viral isu phk buruh gudang garam said iqbal su...,jakarta kompas com presiden konfederasi serika...
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik,"pulau doi, gempa",pulau doi gempa,gempa m 5 3 guncang pulau doi maluku utara,gempa bumi berkekuatan magnitudo m 5 3 menggun...
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik,"pemalsuan emas, emas palsu, polres bengkalis, ...",pemalsuan emas emas palsu polres bengkalis pol...,toko emas palsu di riau dibongkar polisi perhi...,satreskrim polres bengkalis membongkar praktik...
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik,"minyakita, kudus",minyakita kudus,minyakita tak sesuai ukuran juga ditemukan di ...,polisi mendatangi salah satu gudang minyakita ...
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo,"jepang, perdana-menteri, sanae-takaichi, perem...",jepang perdana menteri sanae takaichi perempua...,pimpin ldp sanae takaichi calon kuat pm peremp...,baca berita dengan sedikit iklan klik di sini ...


In [ ]:
kamus_kategori = {
    'Bencana & Cuaca': [
        'gempa', 'tsunami', 'longsor', 'puting beliung', 'cuaca ekstrem',
        'hujan badai', 'erupsi', 'banjir bandang', 'kebakaran hutan',
        'karhutla', 'angin kencang', 'bencana alam', 'bnpb', 'evakuasi',
        'banjir susulan', 'tanggul jebol',
    ],
    'Kesehatan': [
        'rumah sakit', 'puskesmas', 'bpjs', 'pasien', 'dokter', 'obat',
        'wabah', 'stunting', 'demam berdarah', 'covid', 'vaksin',
        'kesehatan', 'gizi buruk', 'nakes', 'tenaga kesehatan',
        'apotek', 'faskes', 'posyandu',
    ],
    'Pendidikan': [
        'sekolah', 'guru', 'spp', 'zonasi', 'ppdb', 'kampus', 'mahasiswa',
        'beasiswa', 'kurikulum', 'ukt', 'siswa', 'pendidikan', 'ujian nasional',
        'sertifikasi guru', 'paud', 'literasi',
    ],
    'Air & Sanitasi': [
        'air bersih', 'pdam', 'krisis air', 'sanitasi', 'toilet', 'mck',
        'air keruh', 'kekeringan', 'air minum', 'sumur', 'irigasi',
        'kebocoran pipa', 'distribusi air',
    ],
    'Transportasi': [
        'macet', 'krl', 'mrt', 'angkot', 'busway', 'transjakarta',
        'stasiun', 'terminal', 'kecelakaan lalu lintas', 'kemacetan',
        'ojek online', 'ojol', 'gojek', 'grab', 'lrt', 'bis kota',
        'angkutan umum', 'parkir liar', 'tilang',
    ],
    'Keamanan & Sosial': [
        'maling', 'begal', 'tawuran', 'kriminalitas', 'polisi', 'keamanan',
        'demo', 'konflik warga', 'pencurian', 'penipuan', 'narkoba',
        'kejahatan', 'kekerasan', 'aksi massa', 'perjudian',
    ],
    'Pelayanan Publik': [
        'ktp', 'kk', 'dukcapil', 'paspor', 'birokrasi', 'pelayanan lambat',
        'pungli', 'sertifikat tanah', 'pelayanan publik', 'antrian panjang',
        'akta lahir', 'administrasi', 'kelurahan', 'kecamatan',
        'perizinan', 'izin usaha',
    ],
    'Infrastruktur & Jalan': [
        'jalan rusak', 'jembatan', 'aspal', 'berlubang', 'proyek jalan',
        'tol', 'gorong', 'trotoar', 'perbaikan jalan', 'jalan berlubang',
        'drainase', 'saluran air', 'infrastruktur', 'pembangunan jalan',
    ],
    'Ekonomi & UMKM': [
        'harga naik', 'sembako', 'inflasi', 'umkm', 'pasar', 'pedagang',
        'bantuan sosial', 'bansos', 'phk', 'pengangguran', 'upah minimum',
        'harga pangan', 'kemiskinan', 'kenaikan harga', 'bbm naik',
        'daya beli', 'ekonomi', 'investasi daerah',
    ],
    'Lingkungan & Sampah': [
        'sampah', 'tps', 'limbah', 'polusi', 'pencemaran', 'banjir',
        'sungai kotor', 'bau', 'berserakan', 'pengelolaan sampah',
        'tempat pembuangan', 'ruang hijau', 'penghijauan', 'illegal dumping',
    ],
}

In [ ]:
def klasifikasi_berita(row):
    # Urutan prioritas: Tags -> Judul -> Isi Konten
    teks_untuk_dicek = [row['clean_tags'], row['clean_judul'], row['clean_konten']]

    for teks in teks_untuk_dicek:
        if teks: # Jika teksnya tidak kosong
            for kategori, kata_kunci in kamus_kategori.items():
                # Jika ada salah satu kata kunci yang cocok di dalam teks
                if any(kunci in teks for kunci in kata_kunci):
                    return kategori

    return 'Lainnya' # Jika tidak ada yang cocok sama sekali

In [ ]:
df['kategori_isu'] = df.apply(klasifikasi_berita, axis=1)

print("Proses pembersihan dan klasifikasi selesai!")

Proses pembersihan dan klasifikasi selesai!


In [ ]:
df[['Judul', 'kategori_isu']].head(10)

,Judul,kategori_isu
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",Ekonomi & UMKM
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara",Bencana & Cuaca
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...",Keamanan & Sosial
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,Pendidikan
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",Keamanan & Sosial
5,Gubernur Banten Terpilih Andra Soni Temui Pres...,Kesehatan
6,Yusril: Pedemo Ditangkap Polisi karena Lakukan...,Keamanan & Sosial
7,"Banjir Besar Landa Jabodetabek, Pimpinan MPR I...",Lingkungan & Sampah
8,"Lagu ""Indonesia Raya"" Kena Royalti? Istana Men...",Lainnya
9,Remaja Ditindak Polisi di Serpong karena Bawa ...,Keamanan & Sosial


In [ ]:
jumlah_kategori = df['kategori_isu'].value_counts()

print("=== Jumlah Data per Kategori ===")
print(jumlah_kategori)

=== Jumlah Data per Kategori ===
kategori_isu
Pendidikan               16388
Keamanan & Sosial        11794
Lainnya                  10462
Kesehatan                 7243
Pelayanan Publik          5848
Ekonomi & UMKM            4358
Transportasi              3532
Bencana & Cuaca           2665
Infrastruktur & Jalan     2501
Lingkungan & Sampah       2101
Air & Sanitasi             489
Name: count, dtype: int64


In [ ]:
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source,all_tags,clean_tags,clean_judul,clean_konten,kategori_isu
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas,"Said Iqbal, industri rokok, PT Gudang Garam, P...",said iqbal industri rokok pt gudang garam phk ...,viral isu phk buruh gudang garam said iqbal su...,jakarta kompas com presiden konfederasi serika...,Ekonomi & UMKM
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik,"pulau doi, gempa",pulau doi gempa,gempa m 5 3 guncang pulau doi maluku utara,gempa bumi berkekuatan magnitudo m 5 3 menggun...,Bencana & Cuaca
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik,"pemalsuan emas, emas palsu, polres bengkalis, ...",pemalsuan emas emas palsu polres bengkalis pol...,toko emas palsu di riau dibongkar polisi perhi...,satreskrim polres bengkalis membongkar praktik...,Keamanan & Sosial
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik,"minyakita, kudus",minyakita kudus,minyakita tak sesuai ukuran juga ditemukan di ...,polisi mendatangi salah satu gudang minyakita ...,Pendidikan
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo,"jepang, perdana-menteri, sanae-takaichi, perem...",jepang perdana menteri sanae takaichi perempua...,pimpin ldp sanae takaichi calon kuat pm peremp...,baca berita dengan sedikit iklan klik di sini ...,Keamanan & Sosial


In [ ]:
data_lainnya = df[df['kategori_isu'] == 'Lainnya']
print(data_lainnya[['Judul', 'clean_konten']].sample(50))

                                                   Judul  \
66254  AC Milan vs Bologna di Final Coppa Italia, Vic...   
35826  600 Relawan Jokowi dan Prabowo Diprediksi Akan...   
80084  Relawan Bakal Arak Jokowi saat Pulang ke Solo ...   
43489  Hasil Final Four Proliga 2025: LavAni Kalahkan...   
4258          Aksi Pasukan Penanggulangan Anti-Teror TNI   
32137                  Budi Arie dalam Vonis Judi Online   
56083  Menteri HAM Ungkap Alasan Prabowo Beri Amnesti...   
76408  Shin Tae-yong Sebut Peluang Timnas Indonesia L...   
63002  Daftar 28 Pemain Timnas Indonesia Dipanggil un...   
43838  Pj Gubernur Teguh Setyabudi Berharap Jakarta B...   
68526  Eks Dirut Terseret Kasus Korupsi Sritex, Bank ...   
1987   Anies Bicara Wujudkan Gagasan Bung Karno saat ...   
37656  Pakai Kemeja Biru Dongker, Jokowi Hadiri Kongr...   
17352  KPK Ungkap Lobi-lobi Asosiasi Haji ke Kemenag ...   
53956  JK Sebut 4 Pulau Masuk Sumut Cacat Formil, Men...   
6536   Prabowo Salami Pengunjung Istiqla